### 🧹 AWS Cleanup Script: Scanned & Deleted Resources

The cleanup script targets **9 core AWS services** across both global and region-specific infrastructure, prioritizing components that incur ongoing hourly or storage charges.

| Service | Specific Resource Scanned | What Gets Deleted |
| :--- | :--- | :--- |
| **S3** *(Global)* | Buckets | Empties all objects, version histories, and delete markers, then deletes the bucket itself. |
| **SageMaker** | Real-time Endpoints | Active inference endpoints incurring hourly compute charges. |
| **SageMaker** | Endpoint Configurations | Saved deployment configurations for model hosting. |
| **SageMaker** | Models | Created SageMaker model entities and container definitions. |
| **EKS** | Managed Kubernetes Clusters | The control plane for EKS clusters. |
| **CloudFormation** | Active Stacks | Automatically updates termination protection to `False` and deletes stack resources (with a `RetainResources` fallback if trapped in `DELETE_FAILED`). |
| **ECR** | Container Repositories | Repositories holding Docker images (forces deletion even if images exist). |
| **Elastic Load Balancing** | Application / Network Load Balancers | Active load balancers (including those auto-provisioned by EKS services). |
| **EC2 / Storage** | Unattached EBS Volumes | Idle block storage volumes sitting in the `available` state. |
| **EC2 / Networking** | Unattached Elastic IPs (EIPs) | Public IPv4 addresses not associated with a running EC2 instance or Load Balancer. |

In [7]:
# =========================================================
# GOOGLE COLAB BASH CELL: AWS CLI ALTERNATIVE
# =========================================================
# Paste this in a separate cell in Google Colab to run the same nuke via AWS CLI

!pip install awscli -q
!pip install boto3

# Set environment variables using Google Colab secrets
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None

os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("AWS_ACCESS_KEY_ID") or userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("AWS_SECRET_ACCESS_KEY") or userdata.get("AWS_SECRET_ACCESS_KEY")

In [8]:
%%bash
echo "🚀 Starting AWS CLI Cleanup Sweep..."

# 1. Delete all S3 Buckets & Contents
echo "------------------------------------------"
echo "Cleaning S3 Buckets..."
echo "------------------------------------------"
for bucket in $(aws s3api list-buckets --query "Buckets[].Name" --output text); do
  echo "❌ Force deleting bucket: $bucket"
  aws s3 rb "s3://$bucket" --force
done

# 2. Iterate through all enabled EC2 regions
REGIONS=$(aws ec2 describe-regions --query "Regions[].RegionName" --output text --region us-east-1)

for region in $REGIONS; do
  echo "------------------------------------------"
  echo "Scanning Region: $region"
  echo "------------------------------------------"

  # Delete SageMaker MLflow Tracking Servers
  SERVERS=$(aws sagemaker list-mlflow-tracking-servers --region "$region" --query "TrackingServerSummaries[].TrackingServerName" --output text 2>/dev/null)
  for server in $SERVERS; do
    if [ "$server" != "None" ] && [ -n "$server" ]; then
      echo "  ❌ Deleting MLflow Server: $server"
      aws sagemaker delete-mlflow-tracking-server --tracking-server-name "$server" --region "$region"
    fi
  done

  # Delete SageMaker Endpoints
  ENDPOINTS=$(aws sagemaker list-endpoints --region "$region" --query "Endpoints[].EndpointName" --output text 2>/dev/null)
  for ep in $ENDPOINTS; do
    if [ "$ep" != "None" ] && [ -n "$ep" ]; then
      echo "  ❌ Deleting Endpoint: $ep"
      aws sagemaker delete-endpoint --endpoint-name "$ep" --region "$region"
    fi
  done

  # Delete SageMaker Endpoint Configs
  CONFIGS=$(aws sagemaker list-endpoint-configs --region "$region" --query "EndpointConfigs[].EndpointConfigName" --output text 2>/dev/null)
  for cfg in $CONFIGS; do
    if [ "$cfg" != "None" ] && [ -n "$cfg" ]; then
      echo "  ❌ Deleting Endpoint Config: $cfg"
      aws sagemaker delete-endpoint-config --endpoint-config-name "$cfg" --region "$region"
    fi
  done

  # Delete SageMaker Models
  MODELS=$(aws sagemaker list-models --region "$region" --query "Models[].ModelName" --output text 2>/dev/null)
  for model in $MODELS; do
    if [ "$model" != "None" ] && [ -n "$model" ]; then
      echo "  ❌ Deleting Model: $model"
      aws sagemaker delete-model --model-name "$model" --region "$region"
    fi
  done

  # Delete EKS Clusters and their Nodegroups
  CLUSTERS=$(aws eks list-clusters --region "$region" --query "clusters" --output text 2>/dev/null)
  for cluster in $CLUSTERS; do
    if [ "$cluster" != "None" ] && [ -n "$cluster" ]; then
      echo "  🔎 Found EKS Cluster: $cluster"

      # Delete Nodegroups associated with the EKS cluster
      NODEGROUPS=$(aws eks list-nodegroups --cluster-name "$cluster" --region "$region" --query "nodegroups" --output text 2>/dev/null)
      for ng in $NODEGROUPS; do
        if [ "$ng" != "None" ] && [ -n "$ng" ]; then
          echo "    ❌ Deleting Nodegroup: $ng from cluster $cluster"
          aws eks delete-nodegroup --cluster-name "$cluster" --nodegroup-name "$ng" --region "$region"
          # Wait for nodegroup deletion to complete before proceeding
          echo "    ⏳ Waiting for nodegroup $ng to be deleted..."
          aws eks wait nodegroup-deleted --cluster-name "$cluster" --nodegroup-name "$ng" --region "$region"
          echo "    ✅ Nodegroup $ng deleted."
        fi
      done

      # Now delete the EKS cluster itself
      echo "  ❌ Deleting EKS Cluster: $cluster"
      aws eks delete-cluster --name "$cluster" --region "$region"
      # EKS cluster deletion can take time. For robust cleanup, consider adding 'aws eks wait cluster-deleted --name "$cluster" --region "$region"'
    fi
  done

done

echo "========================================="
echo "AWS CLI SWEEP COMPLETE!"
echo "========================================="

🚀 Starting AWS CLI Cleanup Sweep...
------------------------------------------
Cleaning S3 Buckets...
------------------------------------------
------------------------------------------
Scanning Region: ap-south-1
------------------------------------------
------------------------------------------
Scanning Region: eu-north-1
------------------------------------------
------------------------------------------
Scanning Region: eu-west-3
------------------------------------------
------------------------------------------
Scanning Region: eu-west-2
------------------------------------------
------------------------------------------
Scanning Region: eu-west-1
------------------------------------------
------------------------------------------
Scanning Region: ap-northeast-3
------------------------------------------
------------------------------------------
Scanning Region: ap-northeast-2
------------------------------------------
------------------------------------------
Scanning 

In [9]:
import os
import time
import boto3
from botocore.exceptions import BotoCoreError, ClientError

# Set DRY_RUN = False to perform actual deletion!
DRY_RUN = False

def get_colab_secret(key_name: str, required: bool = True):
    """Safely retrieves a secret from Google Colab secrets if available."""
    try:
        from google.colab import userdata
        return os.environ.get(key_name) or userdata.get(key_name)
    except Exception:
        if required:
            print(f"⚠️ Warning: Could not retrieve {key_name} from Colab secrets.")
        return None

def get_all_enabled_regions(session=None) -> list:
    """Dynamically retrieves all enabled EC2 regions for the AWS account."""
    try:
        ec2_client = (session or boto3).client("ec2", region_name="us-east-1")
        response = ec2_client.describe_regions(AllRegions=False)
        return [region["RegionName"] for region in response["Regions"]]
    except Exception as exc:
        print(f"⚠️ Could not fetch regions dynamically ({exc}). Falling back to defaults.")
        return ["eu-north-1", "us-east-1", "us-west-2", "eu-west-1"]

def purge_mlflow_servers(sm_client, region_name: str, dry_run: bool = True) -> int:
    """
    Handles MLflow tracking server cleanup across transitional states (Creating, Deleting, etc.).
    Polls actively until all MLflow servers in the region are completely terminated.
    """
    found_count = 0
    try:
        paginator = sm_client.get_paginator("list_mlflow_tracking_servers")
        servers_to_process = []
        for page in paginator.paginate():
            for server in page.get("TrackingServerSummaries", []):
                servers_to_process.append(server)

        if not servers_to_process:
            return 0

        found_count = len(servers_to_process)

        for server in servers_to_process:
            name = server["TrackingServerName"]
            status = server.get("TrackingServerStatus", "UNKNOWN")
            print(f"  ❌ [{region_name}] SageMaker MLflow Server: {name} (Current Status: {status})")

            if dry_run:
                print(f"    🛡️ [DRY-RUN] Would wait for active status and delete MLflow Server: {name}")
                continue

            # Active deletion and state management loop
            max_attempts = 30  # Wait up to 15 minutes (30 * 30s) per server transition
            attempt = 0

            while attempt < max_attempts:
                try:
                    # Refresh server status
                    desc = sm_client.describe_mlflow_tracking_server(TrackingServerName=name)
                    current_status = desc.get("TrackingServerStatus")
                except ClientError as e:
                    # ResourceNotFoundException indicates successful deletion
                    if e.response["Error"]["Code"] in ["ResourceNotFound", "ValidationException"]:
                        print(f"    ✅ MLflow Server '{name}' successfully terminated.")
                        break
                    raise e

                print(f"    ⏳ [{name}] Status: {current_status}. (Attempt {attempt+1}/{max_attempts})")

                if current_status in ["CREATED", "CREATE_FAILED", "UPDATE_FAILED"]:
                    # Disable model registration prior to deletion if active
                    try:
                        sm_client.update_mlflow_tracking_server(
                            TrackingServerName=name,
                            AutomaticModelRegistration=False
                        )
                    except Exception:
                        pass

                    print(f"    🗑️ Requesting deletion for '{name}'...")
                    try:
                        sm_client.delete_mlflow_tracking_server(TrackingServerName=name)
                    except ClientError as exc:
                        print(f"    ⚠️ Delete call failed ({exc.response['Error']['Message']}). Retrying...")

                elif current_status in ["CREATING", "UPDATING"]:
                    print(f"    ⏳ Server is currently {current_status}. Waiting 30s for state transition...")

                elif current_status in ["DELETING"]:
                    print(f"    ⏳ Deletion already in progress by AWS background tasks. Waiting 30s...")

                elif current_status == "DELETE_FAILED":
                    print(f"    ⚠️ Retrying deletion on failed server '{name}'...")
                    try:
                        sm_client.delete_mlflow_tracking_server(TrackingServerName=name)
                    except Exception as exc:
                        print(f"    ❌ Failed to force re-delete: {exc}")

                time.sleep(30)
                attempt += 1

    except Exception as exc:
        print(f"  Error checking MLflow Servers in {region_name}: {exc}")

    return found_count

def purge_eks_clusters(eks_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all EKS clusters and their associated nodegroups in a given region,
    and waits for their termination.
    """
    found_count = 0
    try:
        response = eks_client.list_clusters()
        clusters = response.get("clusters", [])

        if not clusters:
            return 0

        for cluster_name in clusters:
            found_count += 1
            print(f"  🔎 [{region_name}] Found EKS Cluster: {cluster_name}")

            # First, delete all nodegroups associated with the cluster
            try:
                nodegroups_response = eks_client.list_nodegroups(clusterName=cluster_name)
                nodegroups = nodegroups_response.get("nodegroups", [])

                for ng_name in nodegroups:
                    print(f"    ❌ Deleting Nodegroup: {ng_name} from cluster {cluster_name}")
                    if not dry_run:
                        eks_client.delete_nodegroup(clusterName=cluster_name, nodegroupName=ng_name)
                        # Wait for nodegroup to be deleted
                        ng_waiter = eks_client.get_waiter("nodegroup_deleted")
                        print(f"    ⏳ Waiting for Nodegroup '{ng_name}' to be deleted...")
                        ng_waiter.wait(clusterName=cluster_name, nodegroupName=ng_name, WaiterConfig={'Delay': 30, 'MaxAttempts': 60}) # Max 30 minutes
                        print(f"    ✅ Nodegroup '{ng_name}' successfully deleted.")
            except ClientError as e:
                print(f"    ⚠️ Error listing or deleting nodegroups for cluster '{cluster_name}': {e.response['Error']['Message']}")
            except Exception as e:
                print(f"    ⚠️ An unexpected error occurred during nodegroup operations: {e}")

            # After nodegroups are deleted, delete the cluster itself
            if not dry_run:
                try:
                    eks_client.delete_cluster(name=cluster_name)
                    print(f"    🗑️ Requested deletion for EKS Cluster: {cluster_name}")
                    # Wait for cluster to be deleted
                    waiter = eks_client.get_waiter("cluster_deleted")
                    print(f"    ⏳ Waiting for EKS Cluster '{cluster_name}' to be deleted...")
                    waiter.wait(name=cluster_name, WaiterConfig={'Delay': 30, 'MaxAttempts': 60}) # Max 30 minutes
                    print(f"    ✅ EKS Cluster '{cluster_name}' successfully deleted.")
                except ClientError as e:
                    print(f"    ⚠️ Error deleting EKS Cluster '{cluster_name}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during EKS deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping EKS: Invalid credentials.")
        else:
            print(f"  Error listing EKS clusters in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking EKS Clusters in {region_name}: {exc}")
    return found_count

def purge_eks_addons_and_fargate_profiles(eks_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes EKS Fargate profiles and add-ons in a given region.
    """
    found_count = 0
    try:
        clusters_response = eks_client.list_clusters()
        clusters = clusters_response.get("clusters", [])

        for cluster_name in clusters:
            # Fargate Profiles
            fargate_profiles_response = eks_client.list_fargate_profiles(clusterName=cluster_name)
            fargate_profiles = fargate_profiles_response.get("fargateProfileNames", [])
            for fp_name in fargate_profiles:
                found_count += 1
                print(f"  ❌ [{region_name}] Found EKS Fargate Profile: {fp_name} in cluster {cluster_name}")
                if not dry_run:
                    try:
                        eks_client.delete_fargate_profile(clusterName=cluster_name, fargateProfileName=fp_name)
                        print(f"    🗑️ Requested deletion for Fargate Profile: {fp_name}")
                        waiter = eks_client.get_waiter("fargate_profile_deleted")
                        waiter.wait(clusterName=cluster_name, fargateProfileName=fp_name, WaiterConfig={'Delay': 15, 'MaxAttempts': 40})
                        print(f"    ✅ Fargate Profile '{fp_name}' successfully deleted.")
                    except ClientError as e:
                        print(f"    ⚠️ Error deleting Fargate Profile '{fp_name}': {e.response['Error']['Message']}")
                    except Exception as e:
                        print(f"    ⚠️ An unexpected error occurred during Fargate Profile deletion: {e}")

            # EKS Addons
            addons_response = eks_client.list_addons(clusterName=cluster_name)
            addons = addons_response.get("addons", [])
            for addon_name in addons:
                found_count += 1
                print(f"  ❌ [{region_name}] Found EKS Addon: {addon_name} in cluster {cluster_name}")
                if not dry_run:
                    try:
                        eks_client.delete_addon(clusterName=cluster_name, addonName=addon_name)
                        print(f"    🗑️ Requested deletion for EKS Addon: {addon_name}")
                        # Addon deletion doesn't have a direct waiter, might need custom polling
                        # For simplicity, we'll just log the request.
                        print(f"    ✅ EKS Addon '{addon_name}' deletion requested.")
                    except ClientError as e:
                        print(f"    ⚠️ Error deleting EKS Addon '{addon_name}': {e.response['Error']['Message']}")
                    except Exception as e:
                        print(f"    ⚠️ An unexpected error occurred during EKS Addon deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping EKS Addons/Fargate Profiles: Invalid credentials.")
        else:
            print(f"  Error listing EKS Addons/Fargate Profiles in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking EKS Addons/Fargate Profiles in {region_name}: {exc}")
    return found_count

def purge_rds_instances(rds_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all RDS instances in a given region, skipping final snapshots.
    """
    found_count = 0
    try:
        response = rds_client.describe_db_instances()
        instances = response.get("DBInstances", [])

        if not instances:
            return 0

        for instance in instances:
            instance_id = instance["DBInstanceIdentifier"]
            found_count += 1
            print(f"  ❌ [{region_name}] Found RDS Instance: {instance_id}")
            if not dry_run:
                try:
                    rds_client.delete_db_instance(
                        DBInstanceIdentifier=instance_id,
                        SkipFinalSnapshot=True,
                        DeleteAutomatedBackups=True
                    )
                    print(f"    🗑️ Requested deletion for RDS Instance: {instance_id}")
                    waiter = rds_client.get_waiter("db_instance_deleted")
                    print(f"    ⏳ Waiting for RDS Instance '{instance_id}' to be deleted...")
                    waiter.wait(DBInstanceIdentifier=instance_id, WaiterConfig={'Delay': 30, 'MaxAttempts': 60}) # Max 30 minutes
                    print(f"    ✅ RDS Instance '{instance_id}' successfully deleted.")
                except ClientError as e:
                    print(f"    ⚠️ Error deleting RDS Instance '{instance_id}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during RDS deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping RDS: Invalid credentials.")
        else:
            print(f"  Error listing RDS instances in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking RDS Instances in {region_name}: {exc}")
    return found_count

def purge_dynamodb_tables(dynamodb_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all DynamoDB tables in a given region.
    """
    found_count = 0
    try:
        response = dynamodb_client.list_tables()
        tables = response.get("TableNames", [])

        if not tables:
            return 0

        for table_name in tables:
            found_count += 1
            print(f"  ❌ [{region_name}] Found DynamoDB Table: {table_name}")
            if not dry_run:
                try:
                    dynamodb_client.delete_table(TableName=table_name)
                    print(f"    🗑️ Requested deletion for DynamoDB Table: {table_name}")
                    waiter = dynamodb_client.get_waiter("table_not_exists")
                    print(f"    ⏳ Waiting for DynamoDB Table '{table_name}' to be deleted...")
                    waiter.wait(TableName=table_name, WaiterConfig={'Delay': 10, 'MaxAttempts': 60}) # Max 10 minutes
                    print(f"    ✅ DynamoDB Table '{table_name}' successfully deleted.")
                except ClientError as e:
                    print(f"    ⚠️ Error deleting DynamoDB Table '{table_name}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during DynamoDB deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping DynamoDB: Invalid credentials.")
        else:
            print(f"  Error listing DynamoDB tables in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking DynamoDB Tables in {region_name}: {exc}")
    return found_count

def purge_ec2_instances(ec2_client, region_name: str, dry_run: bool = True) -> int:
    """
    Terminates all running/stopped EC2 instances and deletes associated volumes and snapshots.
    """
    found_count = 0
    try:
        # Terminate EC2 instances
        instances_response = ec2_client.describe_instances(
            Filters=[
                {'Name': 'instance-state-name', 'Values': ['running', 'stopped', 'stopping']}
            ]
        )
        instance_ids = []
        for reservation in instances_response.get("Reservations", []):
            for instance in reservation.get("Instances", []):
                instance_ids.append(instance["InstanceId"])

        if instance_ids:
            found_count += len(instance_ids)
            print(f"  ❌ [{region_name}] Found EC2 Instances: {', '.join(instance_ids)}")
            if not dry_run:
                try:
                    ec2_client.terminate_instances(InstanceIds=instance_ids)
                    print(f"    🗑️ Requested termination for EC2 Instances: {', '.join(instance_ids)}")
                    waiter = ec2_client.get_waiter("instance_terminated")
                    print(f"    ⏳ Waiting for EC2 Instances '{', '.join(instance_ids)}' to terminate...")
                    waiter.wait(InstanceIds=instance_ids, WaiterConfig={'Delay': 15, 'MaxAttempts': 40}) # Max 10 minutes
                    print(f"    ✅ EC2 Instances '{', '.join(instance_ids)}' successfully terminated.")
                except ClientError as e:
                    print(f"    ⚠️ Error terminating EC2 Instances: {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during EC2 termination: {e}")

        # Delete unattached EBS volumes (attached volumes will be deleted with instances)
        volumes_response = ec2_client.describe_volumes(
            Filters=[{'Name': 'status', 'Values': ['available']}]
        )
        for volume in volumes_response.get("Volumes", []):
            volume_id = volume["VolumeId"]
            found_count += 1
            print(f"  ❌ [{region_name}] Found unattached EBS Volume: {volume_id}")
            if not dry_run:
                try:
                    ec2_client.delete_volume(VolumeId=volume_id)
                    print(f"    ✅ Deleted unattached EBS Volume: {volume_id}")
                except ClientError as e:
                    print(f"    ⚠️ Error deleting EBS Volume '{volume_id}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during EBS volume deletion: {e}")

        # Delete EBS snapshots
        snapshots_response = ec2_client.describe_snapshots(OwnerIds=['self'])
        for snapshot in snapshots_response.get("Snapshots", []):
            snapshot_id = snapshot["SnapshotId"]
            found_count += 1
            print(f"  ❌ [{region_name}] Found EBS Snapshot: {snapshot_id}")
            if not dry_run:
                try:
                    ec2_client.delete_snapshot(SnapshotId=snapshot_id)
                    print(f"    ✅ Deleted EBS Snapshot: {snapshot_id}")
                except ClientError as e:
                    print(f"    ⚠️ Error deleting EBS Snapshot '{snapshot_id}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during EBS snapshot deletion: {e}")

        # Delete AMIs (images)
        images_response = ec2_client.describe_images(Owners=['self'])
        for image in images_response.get("Images", []):
            image_id = image["ImageId"]
            found_count += 1
            print(f"  ❌ [{region_name}] Found AMI: {image_id} ({image['Name']})")
            if not dry_run:
                try:
                    ec2_client.deregister_image(ImageId=image_id)
                    print(f"    ✅ Deregistered AMI: {image_id}")
                except ClientError as e:
                    print(f"    ⚠️ Error deregistering AMI '{image_id}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during AMI deregistration: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping EC2: Invalid credentials.")
        else:
            print(f"  Error listing EC2 resources in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking EC2 resources in {region_name}: {exc}")
    return found_count

def purge_eip_addresses(ec2_client, region_name: str, dry_run: bool = True) -> int:
    """
    Releases all unattached Elastic IP (EIP) addresses in a given region.
    """
    found_count = 0
    try:
        response = ec2_client.describe_addresses()
        for eip in response.get("Addresses", []):
            if "InstanceId" not in eip and "NetworkInterfaceId" not in eip: # Check if EIP is unattached
                allocation_id = eip["AllocationId"]
                found_count += 1
                print(f"  ❌ [{region_name}] Found unattached EIP: {eip['PublicIp']} (Allocation ID: {allocation_id})")
                if not dry_run:
                    try:
                        ec2_client.release_address(AllocationId=allocation_id)
                        print(f"    ✅ Released EIP: {eip['PublicIp']}")
                    except ClientError as e:
                        print(f"    ⚠️ Error releasing EIP '{eip['PublicIp']}': {e.response['Error']['Message']}")
                    except Exception as e:
                        print(f"    ⚠️ An unexpected error occurred during EIP release: {e}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping EIPs: Invalid credentials.")
        else:
            print(f"  Error listing EIPs in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking EIPs in {region_name}: {exc}")
    return found_count

def purge_nat_gateways(ec2_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all NAT Gateways in a given region.
    """
    found_count = 0
    try:
        response = ec2_client.describe_nat_gateways()
        for nat_gateway in response.get("NatGateways", []):
            nat_gateway_id = nat_gateway["NatGatewayId"]
            if nat_gateway["State"] in ["pending", "available", "failed"]:
                found_count += 1
                print(f"  ❌ [{region_name}] Found NAT Gateway: {nat_gateway_id} (State: {nat_gateway['State']})")
                if not dry_run:
                    try:
                        ec2_client.delete_nat_gateway(NatGatewayId=nat_gateway_id)
                        print(f"    🗑️ Requested deletion for NAT Gateway: {nat_gateway_id}")
                        # NAT Gateway deletion can take time, no direct waiter in boto3 for 'deleted'
                        # We'll just log the request.
                        print(f"    ✅ NAT Gateway '{nat_gateway_id}' deletion requested.")
                    except ClientError as e:
                        print(f"    ⚠️ Error deleting NAT Gateway '{nat_gateway_id}': {e.response['Error']['Message']}")
                    except Exception as e:
                        print(f"    ⚠️ An unexpected error occurred during NAT Gateway deletion: {e}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping NAT Gateways: Invalid credentials.")
        else:
            print(f"  Error listing NAT Gateways in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking NAT Gateways in {region_name}: {exc}")
    return found_count

def purge_lambda_functions(lambda_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all Lambda functions in a given region.
    """
    found_count = 0
    try:
        paginator = lambda_client.get_paginator("list_functions")
        for page in paginator.paginate():
            for func in page.get("Functions", []):
                func_name = func["FunctionName"]
                found_count += 1
                print(f"  ❌ [{region_name}] Found Lambda Function: {func_name}")
                if not dry_run:
                    try:
                        lambda_client.delete_function(FunctionName=func_name)
                        print(f"    ✅ Deleted Lambda Function: {func_name}")
                    except ClientError as e:
                        print(f"    ⚠️ Error deleting Lambda Function '{func_name}': {e.response['Error']['Message']}")
                    except Exception as e:
                        print(f"    ⚠️ An unexpected error occurred during Lambda deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping Lambda: Invalid credentials.")
        else:
            print(f"  Error listing Lambda functions in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking Lambda Functions in {region_name}: {exc}")
    return found_count

def purge_sqs_queues(sqs_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all SQS queues in a given region.
    """
    found_count = 0
    try:
        response = sqs_client.list_queues()
        queue_urls = response.get("QueueUrls", [])

        if not queue_urls:
            return 0

        for queue_url in queue_urls:
            found_count += 1
            print(f"  ❌ [{region_name}] Found SQS Queue: {queue_url}")
            if not dry_run:
                try:
                    sqs_client.delete_queue(QueueUrl=queue_url)
                    print(f"    ✅ Deleted SQS Queue: {queue_url}")
                except ClientError as e:
                    print(f"    ⚠️ Error deleting SQS Queue '{queue_url}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during SQS deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping SQS: Invalid credentials.")
        else:
            print(f"  Error listing SQS queues in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking SQS Queues in {region_name}: {exc}")
    return found_count

def purge_sns_topics(sns_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all SNS topics in a given region.
    """
    found_count = 0
    try:
        paginator = sns_client.get_paginator("list_topics")
        for page in paginator.paginate():
            for topic in page.get("Topics", []):
                topic_arn = topic["TopicArn"]
                found_count += 1
                print(f"  ❌ [{region_name}] Found SNS Topic: {topic_arn}")
                if not dry_run:
                    try:
                        sns_client.delete_topic(TopicArn=topic_arn)
                        print(f"    ✅ Deleted SNS Topic: {topic_arn}")
                    except ClientError as e:
                        print(f"    ⚠️ Error deleting SNS Topic '{topic_arn}': {e.response['Error']['Message']}")
                    except Exception as e:
                        print(f"    ⚠️ An unexpected error occurred during SNS deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping SNS: Invalid credentials.")
        else:
            print(f"  Error listing SNS topics in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking SNS Topics in {region_name}: {exc}")
    return found_count

def purge_ecs_resources(ecs_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all ECS clusters, services, and task definitions in a given region.
    """
    found_count = 0
    try:
        # Delete Services first
        clusters = ecs_client.list_clusters().get("clusterArns", [])
        for cluster_arn in clusters:
            cluster_name = cluster_arn.split('/')[-1]
            services = ecs_client.list_services(cluster=cluster_name).get("serviceArns", [])
            for service_arn in services:
                service_name = service_arn.split('/')[-1]
                found_count += 1
                print(f"  ❌ [{region_name}] Found ECS Service: {service_name} in cluster {cluster_name}")
                if not dry_run:
                    try:
                        ecs_client.update_service(cluster=cluster_name, service=service_name, desiredCount=0)
                        print(f"    🗑️ Scaling down ECS Service '{service_name}'...")
                        ecs_client.delete_service(cluster=cluster_name, service=service_name, force=True)
                        print(f"    ✅ Deleted ECS Service: {service_name}")
                    except ClientError as e:
                        print(f"    ⚠️ Error deleting ECS Service '{service_name}': {e.response['Error']['Message']}")
                    except Exception as e:
                        print(f"    ⚠️ An unexpected error occurred during ECS service deletion: {e}")

        # Delete Task Definitions
        task_defs = ecs_client.list_task_definitions().get("taskDefinitionArns", [])
        for td_arn in task_defs:
            found_count += 1
            print(f"  ❌ [{region_name}] Found ECS Task Definition: {td_arn}")
            if not dry_run:
                try:
                    ecs_client.deregister_task_definition(taskDefinition=td_arn)
                    print(f"    ✅ Deregistered ECS Task Definition: {td_arn}")
                except ClientError as e:
                    print(f"    ⚠️ Error deregistering ECS Task Definition '{td_arn}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during ECS task definition deregistration: {e}")

        # Delete Clusters
        for cluster_arn in clusters:
            cluster_name = cluster_arn.split('/')[-1]
            found_count += 1
            print(f"  ❌ [{region_name}] Found ECS Cluster: {cluster_name}")
            if not dry_run:
                try:
                    ecs_client.delete_cluster(cluster=cluster_name)
                    print(f"    ✅ Deleted ECS Cluster: {cluster_name}")
                except ClientError as e:
                    print(f"    ⚠️ Error deleting ECS Cluster '{cluster_name}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during ECS cluster deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping ECS: Invalid credentials.")
        else:
            print(f"  Error listing ECS resources in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking ECS Resources in {region_name}: {exc}")
    return found_count

def purge_elbs(elb_client, elbv2_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all Elastic Load Balancers (Classic, Application, Network) in a given region.
    """
    found_count = 0
    try:
        # Classic Load Balancers
        response = elb_client.describe_load_balancers()
        for lb in response.get("LoadBalancerDescriptions", []):
            lb_name = lb["LoadBalancerName"]
            found_count += 1
            print(f"  ❌ [{region_name}] Found Classic ELB: {lb_name}")
            if not dry_run:
                try:
                    elb_client.delete_load_balancer(LoadBalancerName=lb_name)
                    print(f"    ✅ Deleted Classic ELB: {lb_name}")
                except ClientError as e:
                    print(f"    ⚠️ Error deleting Classic ELB '{lb_name}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during Classic ELB deletion: {e}")

        # Application and Network Load Balancers
        response = elbv2_client.describe_load_balancers()
        for lb in response.get("LoadBalancers", []):
            lb_arn = lb["LoadBalancerArn"]
            lb_name = lb["LoadBalancerName"]
            found_count += 1
            print(f"  ❌ [{region_name}] Found ALB/NLB: {lb_name}")
            if not dry_run:
                try:
                    elbv2_client.delete_load_balancer(LoadBalancerArn=lb_arn)
                    print(f"    ✅ Deleted ALB/NLB: {lb_name}")
                except ClientError as e:
                    print(f"    ⚠️ Error deleting ALB/NLB '{lb_name}': {e.response['Error']['Message']}")
                except Exception as e:
                    print(f"    ⚠️ An unexpected error occurred during ALB/NLB deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping ELB: Invalid credentials.")
        else:
            print(f"  Error listing ELBs in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking ELBs in {region_name}: {exc}")
    return found_count

def purge_cloudformation_stacks(cf_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes CloudFormation stacks in a region. Automatically handles stacks in
    DELETE_FAILED state by identifying and retaining blocked resources.
    """
    found_count = 0
    try:
        paginator = cf_client.get_paginator("list_stacks")
        for page in paginator.paginate(StackStatusFilter=[
            'CREATE_COMPLETE', 'UPDATE_COMPLETE', 'ROLLBACK_COMPLETE',
            'CREATE_FAILED', 'ROLLBACK_FAILED', 'UPDATE_ROLLBACK_FAILED', 'DELETE_FAILED'
        ]):
            for stack_summary in page.get("StackSummaries", []):
                stack_name = stack_summary["StackName"]
                stack_status = stack_summary["StackStatus"]

                if stack_status == 'DELETE_COMPLETE':
                    continue

                found_count += 1
                print(f"  ❌ [{region_name}] Found CloudFormation Stack: {stack_name} (Status: {stack_status})")

                if dry_run:
                    print(f"    🛡️ [DRY-RUN] Would delete stack: {stack_name}")
                    continue

                # 1. Disable Termination Protection
                try:
                    cf_client.update_termination_protection(
                        StackName=stack_name,
                        EnableTerminationProtection=False
                    )
                    print(f"    🗑️ Disabled termination protection for stack '{stack_name}'.")
                except ClientError as e:
                    err_msg = e.response['Error']['Message']
                    if not any(ign in err_msg for ign in ["cannot be updated", "does not exist", "Resource not found"]):
                        print(f"    ⚠️ Error disabling termination protection: {err_msg}")

                # 2. Handle stacks ALREADY in DELETE_FAILED vs normal stacks
                retained_resources = []
                if stack_status == 'DELETE_FAILED':
                    print(f"    ⚠️ Stack '{stack_name}' is already in DELETE_FAILED state. Fetching failed resources...")
                    try:
                        resources = cf_client.describe_stack_resources(StackName=stack_name)['StackResources']
                        retained_resources = [
                            res['LogicalResourceId'] for res in resources
                            if res['ResourceStatus'] in ['DELETE_FAILED', 'CREATE_COMPLETE', 'UPDATE_COMPLETE']
                        ]
                        print(f"    📌 Resources to retain: {', '.join(retained_resources)}")
                    except Exception as e:
                        print(f"    ⚠️ Could not fetch stack resources: {e}")

                # 3. Issue Delete Stack Request
                print(f"    🗑️ Requesting deletion for CloudFormation Stack: {stack_name}")
                if retained_resources:
                    cf_client.delete_stack(StackName=stack_name, RetainResources=retained_resources)
                else:
                    cf_client.delete_stack(StackName=stack_name)

                # 4. Wait for Deletion Completion
                waiter = cf_client.get_waiter("stack_delete_complete")
                print(f"    ⏳ Waiting for CloudFormation Stack '{stack_name}' to be deleted...")

                try:
                    waiter.wait(StackName=stack_name, WaiterConfig={'Delay': 20, 'MaxAttempts': 90})
                    print(f"    ✅ CloudFormation Stack '{stack_name}' successfully deleted.")
                except ClientError as waiter_err:
                    # Catch the waiter failure if it enters DELETE_FAILED during execution
                    print(f"    ⚠️ Waiter caught failure: {waiter_err}")
                    print(f"    🔄 Retrying deletion with RetainResources...")

                    try:
                        resources = cf_client.describe_stack_resources(StackName=stack_name)['StackResources']
                        failed_ids = [
                            res['LogicalResourceId'] for res in resources
                            if res['ResourceStatus'] not in ['DELETE_COMPLETE', 'DELETE_IN_PROGRESS']
                        ]

                        if failed_ids:
                            print(f"    📌 Force deleting while retaining: {', '.join(failed_ids)}")
                            cf_client.delete_stack(StackName=stack_name, RetainResources=failed_ids)
                            waiter.wait(StackName=stack_name, WaiterConfig={'Delay': 20, 'MaxAttempts': 90})
                            print(f"    ✅ CloudFormation Stack '{stack_name}' successfully deleted (retained stuck resources).")
                    except Exception as force_err:
                        print(f"    ❌ Failed to force-delete stack '{stack_name}': {force_err}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping CloudFormation: Invalid credentials.")
        else:
            print(f"  Error listing CloudFormation stacks in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking CloudFormation Stacks in {region_name}: {exc}")

    return found_count

def purge_ecr_repositories(ecr_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all ECR repositories in a given region, forcing deletion even if images exist.
    """
    found_count = 0
    try:
        paginator = ecr_client.get_paginator("describe_repositories")
        for page in paginator.paginate():
            for repo in page.get("repositories", []):
                repo_name = repo["repositoryName"]
                found_count += 1
                print(f"  ❌ [{region_name}] Found ECR Repository: {repo_name}")
                if not dry_run:
                    try:
                        ecr_client.delete_repository(repositoryName=repo_name, force=True)
                        print(f"    ✅ Deleted ECR Repository: {repo_name}")
                    except ClientError as e:
                        print(f"    ⚠️ Error deleting ECR Repository '{repo_name}': {e.response['Error']['Message']}")
                    except Exception as e:
                        print(f"    ⚠️ An unexpected error occurred during ECR deletion: {e}")

    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidClientTokenId":
            print(f"  [{region_name}] Skipping ECR: Invalid credentials.")
        else:
            print(f"  Error listing ECR repositories in {region_name}: {e}")
    except Exception as exc:
        print(f"  Error checking ECR Repositories in {region_name}: {exc}")
    return found_count

def purge_vpc_resources(ec2_client, region_name: str, dry_run: bool = True) -> int:
    """
    Deletes all non-default VPCs and associated resources in a given region.
    """
    found_count = 0
    try:
        vpcs_response = ec2_client.describe_vpcs()
        vpcs_to_delete = [vpc['VpcId'] for vpc in vpcs_response.get('Vpcs', []) if not vpc.get('IsDefault', False)]

        for vpc_id in vpcs_to_delete:
            print(f"  🔎 [{region_name}] Processing VPC: {vpc_id}")

            # 1. Delete NAT Gateways
            nat_gws = ec2_client.describe_nat_gateways(Filter=[{'Name': 'vpc-id', 'Values': [vpc_id]}]).get('NatGateways', [])
            for nat in nat_gws:
                if nat['State'] not in ['deleting', 'deleted']:
                    found_count += 1
                    print(f"    ❌ Deleting NAT Gateway: {nat['NatGatewayId']}")
                    if not dry_run:
                        ec2_client.delete_nat_gateway(NatGatewayId=nat['NatGatewayId'])

            # Wait briefly if NAT Gateways were deleted to allow transition
            if nat_gws and not dry_run:
                print("    ⏳ Waiting for NAT Gateways to delete...")
                time.sleep(15)

            # 2. Detach and delete Internet Gateways
            igws = ec2_client.describe_internet_gateways(Filters=[{'Name': 'attachment.vpc-id', 'Values': [vpc_id]}]).get('InternetGateways', [])
            for igw in igws:
                igw_id = igw['InternetGatewayId']
                found_count += 1
                print(f"    ❌ Detaching/Deleting IGW: {igw_id}")
                if not dry_run:
                    try:
                        ec2_client.detach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
                        ec2_client.delete_internet_gateway(InternetGatewayId=igw_id)
                    except ClientError as e:
                        print(f"      ⚠️ Error deleting IGW '{igw_id}': {e.response['Error']['Message']}")

            # 3. Delete VPC Endpoints
            endpoints = ec2_client.describe_vpc_endpoints(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]).get('VpcEndpoints', [])
            for ep in endpoints:
                found_count += 1
                print(f"    ❌ Deleting VPC Endpoint: {ep['VpcEndpointId']}")
                if not dry_run:
                    try:
                        ec2_client.delete_vpc_endpoints(VpcEndpointIds=[ep['VpcEndpointId']])
                    except ClientError as e:
                        print(f"      ⚠️ Error deleting Endpoint '{ep['VpcEndpointId']}': {e.response['Error']['Message']}")

            # 4. Detach/Delete Network Interfaces (ENIs)
            enis = ec2_client.describe_network_interfaces(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]).get('NetworkInterfaces', [])
            for eni in enis:
                eni_id = eni['NetworkInterfaceId']
                found_count += 1
                print(f"    ❌ Detaching/Deleting ENI: {eni_id}")
                if not dry_run:
                    try:
                        if eni.get('Attachment'):
                            ec2_client.detach_network_interface(AttachmentId=eni['Attachment']['AttachmentId'], Force=True)
                            time.sleep(2)
                        ec2_client.delete_network_interface(NetworkInterfaceId=eni_id)
                    except ClientError as e:
                        print(f"      ⚠️ Error deleting ENI '{eni_id}': {e.response['Error']['Message']}")

            # 5. Revoke Security Group Rules (Breaks cross-references)
            sgs = ec2_client.describe_security_groups(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]).get('SecurityGroups', [])
            if not dry_run:
                for sg in sgs:
                    if sg['GroupName'] != 'default':
                        # Clear ingress
                        if sg.get('IpPermissions'):
                            ec2_client.revoke_security_group_ingress(GroupId=sg['GroupId'], IpPermissions=sg['IpPermissions'])
                        # Clear egress
                        if sg.get('IpPermissionsEgress'):
                            ec2_client.revoke_security_group_egress(GroupId=sg['GroupId'], IpPermissions=sg['IpPermissionsEgress'])

            # 6. Delete Security Groups (Non-default)
            for sg in sgs:
                if sg['GroupName'] != 'default':
                    sg_id = sg['GroupId']
                    found_count += 1
                    print(f"    ❌ Deleting Security Group: {sg_id}")
                    if not dry_run:
                        try:
                            ec2_client.delete_security_group(GroupId=sg_id)
                            print(f"      ✅ Deleted Security Group: {sg_id}")
                        except ClientError as e:
                            print(f"      ⚠️ Error deleting SG '{sg_id}': {e.response['Error']['Message']}")

            # 7. Delete Subnets
            subnets = ec2_client.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]).get('Subnets', [])
            for subnet in subnets:
                sub_id = subnet['SubnetId']
                found_count += 1
                print(f"    ❌ Deleting Subnet: {sub_id}")
                if not dry_run:
                    try:
                        ec2_client.delete_subnet(SubnetId=sub_id)
                        print(f"      ✅ Deleted Subnet: {sub_id}")
                    except ClientError as e:
                        print(f"      ⚠️ Error deleting Subnet '{sub_id}': {e.response['Error']['Message']}")

            # 8. Delete Custom Route Tables
            route_tables = ec2_client.describe_route_tables(Filters=[{'Name': 'vpc-id', 'Values': [vpc_id]}]).get('RouteTables', [])
            for rt in route_tables:
                is_main = any(assoc.get('Main', False) for assoc in rt.get('Associations', []))
                if not is_main:
                    rt_id = rt['RouteTableId']
                    found_count += 1
                    print(f"    ❌ Deleting Route Table: {rt_id}")
                    if not dry_run:
                        try:
                            for assoc in rt.get('Associations', []):
                                if 'RouteTableAssociationId' in assoc:
                                    ec2_client.disassociate_route_table(AssociationId=assoc['RouteTableAssociationId'])
                            ec2_client.delete_route_table(RouteTableId=rt_id)
                        except ClientError as e:
                            print(f"      ⚠️ Error deleting Route Table '{rt_id}': {e.response['Error']['Message']}")

            # 9. Delete VPC
            print(f"    ❌ Deleting VPC: {vpc_id}")
            if not dry_run:
                try:
                    ec2_client.delete_vpc(VpcId=vpc_id)
                    print(f"      ✅ Deleted VPC: {vpc_id}")
                except ClientError as e:
                    print(f"      ⚠️ Error deleting VPC '{vpc_id}': {e.response['Error']['Message']}")

    except Exception as exc:
        print(f"  Error processing VPC resources in {region_name}: {exc}")

    return found_count

def nuke_all_aws_resources(dry_run: bool = True):
    if dry_run:
        print("🛡️  RUNNING IN DRY-RUN MODE: No resources will be deleted.\n")
    else:
        print("💥 CRITICAL WARNING: DRY-RUN IS DISABLED. Wiping ALL detected resources!\n")

    access_key = os.environ.get("AWS_ACCESS_KEY_ID") or get_colab_secret("AWS_ACCESS_KEY_ID")
    secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY") or get_colab_secret("AWS_SECRET_ACCESS_KEY")
    session_token = os.environ.get("AWS_SESSION_TOKEN") or get_colab_secret("AWS_SESSION_TOKEN", required=False)

    init_session = boto3.session.Session(
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        aws_session_token=session_token,
        region_name="us-east-1",
    )

    regions = get_all_enabled_regions(session=init_session)
    total_found = 0

    # Clean up S3 buckets first (global service, but accessed via a region)
    try:
        s3_client = init_session.client("s3")
        s3_resource = init_session.resource("s3")
        buckets = s3_client.list_buckets().get("Buckets", [])
        for b in buckets:
            name = b["Name"]
            print(f"  ❌ Found S3 Bucket: {name}")
            total_found += 1
            if not dry_run:
                bucket = s3_resource.Bucket(name)
                bucket.object_versions.delete() # Delete all object versions
                bucket.delete() # Delete the bucket itself
                print(f"    ✅ Deleted S3 Bucket: {name}")
    except Exception as exc:
        print(f"  Error checking S3 Buckets: {exc}")

    # Region sweep
    for region_name in regions:
        print(f"\n==========================================")
        print(f"Scanning Region: {region_name}")
        print(f"==========================================")
        try:
            session = boto3.session.Session(
                region_name=region_name,
                aws_access_key_id=access_key,
                aws_secret_access_key=secret_key,
                aws_session_token=session_token,
            )
            # Initialize clients for various services
            sm_client = session.client("sagemaker", region_name=region_name)
            eks_client = session.client("eks", region_name=region_name)
            rds_client = session.client("rds", region_name=region_name)
            dynamodb_client = session.client("dynamodb", region_name=region_name)
            ec2_client = session.client("ec2", region_name=region_name)
            lambda_client = session.client("lambda", region_name=region_name)
            sqs_client = session.client("sqs", region_name=region_name)
            sns_client = session.client("sns", region_name=region_name)
            ecs_client = session.client("ecs", region_name=region_name)
            elb_client = session.client("elb", region_name=region_name) # Classic ELB
            elbv2_client = session.client("elbv2", region_name=region_name) # ALB/NLB
            cf_client = session.client("cloudformation", region_name=region_name)
            ecr_client = session.client("ecr", region_name=region_name)

            # --- SageMaker Resources ---
            total_found += purge_mlflow_servers(sm_client, region_name, dry_run=dry_run)
            for page in sm_client.get_paginator("list_endpoints").paginate():
                for ep in page.get("Endpoints", []):
                    total_found += 1
                    print(f"  ❌ [{region_name}] Found SageMaker Endpoint: {ep['EndpointName']}")
                    if not dry_run:
                        sm_client.delete_endpoint(EndpointName=ep["EndpointName"])

            for page in sm_client.get_paginator("list_endpoint_configs").paginate():
                for cfg in page.get("EndpointConfigs", []):
                    total_found += 1
                    print(f"  ❌ [{region_name}] Found SageMaker Endpoint Config: {cfg['EndpointConfigName']}")
                    if not dry_run:
                        sm_client.delete_endpoint_config(EndpointConfigName=cfg["EndpointConfigName"])

            for page in sm_client.get_paginator("list_models").paginate():
                for model in page.get("Models", []):
                    total_found += 1
                    print(f"  ❌ [{region_name}] Found SageMaker Model: {model['ModelName']}")
                    if not dry_run:
                        sm_client.delete_model(ModelName=model["ModelName"])

            # --- EKS Clusters ---
            total_found += purge_eks_clusters(eks_client, region_name, dry_run=dry_run)
            total_found += purge_eks_addons_and_fargate_profiles(eks_client, region_name, dry_run=dry_run)

            # --- RDS Instances ---
            total_found += purge_rds_instances(rds_client, region_name, dry_run=dry_run)

            # --- DynamoDB Tables ---
            total_found += purge_dynamodb_tables(dynamodb_client, region_name, dry_run=dry_run)

            # --- EC2 Resources (Instances, Volumes, Snapshots, AMIs) ---
            total_found += purge_ec2_instances(ec2_client, region_name, dry_run=dry_run)
            total_found += purge_eip_addresses(ec2_client, region_name, dry_run=dry_run)
            total_found += purge_nat_gateways(ec2_client, region_name, dry_run=dry_run)

            # --- Lambda Functions ---
            total_found += purge_lambda_functions(lambda_client, region_name, dry_run=dry_run)

            # --- SQS Queues ---
            total_found += purge_sqs_queues(sqs_client, region_name, dry_run=dry_run)

            # --- SNS Topics ---
            total_found += purge_sns_topics(sns_client, region_name, dry_run=dry_run)

            # --- ECS Resources (Clusters, Services, Task Definitions) ---
            total_found += purge_ecs_resources(ecs_client, region_name, dry_run=dry_run)

            # --- Elastic Load Balancers (ELB, ALB, NLB) ---
            total_found += purge_elbs(elb_client, elbv2_client, region_name, dry_run=dry_run)

            # --- CloudFormation Stacks ---
            total_found += purge_cloudformation_stacks(cf_client, region_name, dry_run=dry_run)

            # --- ECR Repositories ---
            total_found += purge_ecr_repositories(ecr_client, region_name, dry_run=dry_run)

            # --- VPC Resources (VPCs, Subnets, IGWs, Route Tables, Security Groups, Endpoints) ---
            total_found += purge_vpc_resources(ec2_client, region_name, dry_run=dry_run)

        except Exception as exc:
            print(f"[{region_name}] Access or client initialization failed: {exc}")

    print(f"\n==========================================")
    print(f"SWEEP COMPLETE: Processed {total_found} total resource(s).")
    print(f"==========================================")

# Execute sweep
nuke_all_aws_resources(dry_run=DRY_RUN)


💥 CRITICAL WARNING: DRY-RUN IS DISABLED. Wiping ALL detected resources!


Scanning Region: ap-south-1

Scanning Region: eu-north-1

Scanning Region: eu-west-3

Scanning Region: eu-west-2

Scanning Region: eu-west-1

Scanning Region: ap-northeast-3

Scanning Region: ap-northeast-2

Scanning Region: ap-northeast-1

Scanning Region: ca-central-1

Scanning Region: sa-east-1

Scanning Region: ap-southeast-1

Scanning Region: ap-southeast-2

Scanning Region: eu-central-1

Scanning Region: us-east-1

Scanning Region: us-east-2

Scanning Region: us-west-1

Scanning Region: us-west-2

SWEEP COMPLETE: Processed 0 total resource(s).


In [10]:
import os
import boto3

def get_bucket_region(s3_client, bucket_name: str) -> str:
    """Detect the exact AWS region where an S3 bucket resides."""
    try:
        location = s3_client.get_bucket_location(Bucket=bucket_name).get("LocationConstraint")
        # AWS returns None for us-east-1 and "EU" for legacy eu-west-1
        if location is None:
            return "us-east-1"
        if location == "EU":
            return "eu-west-1"
        return location
    except Exception:
        return "us-east-1"

def delete_usecase_etl_buckets(dry_run: bool = True):
    try:
        access_key = os.environ.get("AWS_ACCESS_KEY_ID") or get_colab_secret("AWS_ACCESS_KEY_ID")
        secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY") or get_colab_secret("AWS_SECRET_ACCESS_KEY")
        session_token = os.environ.get("AWS_SESSION_TOKEN") or get_colab_secret("AWS_SESSION_TOKEN", required=False)

        session = boto3.session.Session(
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            aws_session_token=session_token
        )
    except Exception as exc:
        print(f"❌ Could not build AWS session: {exc}")
        return

    # Global client used to scan account buckets across all regions
    global_s3_client = session.client("s3")

    try:
        account_id = session.client("sts").get_caller_identity()["Account"]
    except Exception as exc:
        print(f"❌ Could not resolve AWS account id: {exc}")
        return

    expected = {f"usecase-etl-{i}-{account_id}" for i in (1, 2)}
    buckets = global_s3_client.list_buckets().get("Buckets", [])
    targets = [b["Name"] for b in buckets if b["Name"] in expected or b["Name"].startswith("usecase-etl-")]

    print(f"🗂️ Found {len(targets)} UseCase ETL bucket(s) to remove across all regions.")

    for name in targets:
        # Dynamically resolve region for each bucket before deletion operations
        bucket_region = get_bucket_region(global_s3_client, name)
        print(f"📍 Target '{name}' resides in region: {bucket_region}")

        # Instantiate region-specific resource to execute deletion cleanly
        regional_s3_resource = session.resource("s3", region_name=bucket_region)
        empty_and_delete_s3_bucket(regional_s3_resource, name, dry_run=dry_run)

    print(f"✅ UseCase ETL bucket cleanup complete ({len(targets)} bucket(s) handled).")

delete_usecase_etl_buckets(dry_run=DRY_RUN)

🗂️ Found 0 UseCase ETL bucket(s) to remove across all regions.
✅ UseCase ETL bucket cleanup complete (0 bucket(s) handled).


In [11]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-23 11:03:13
